# RAG Evaluation — DiabetesAssist
Tests the RAG pipeline with diabetes questions and scores answers using LLM-as-Judge.

In [ ]:
import sys
sys.path.append('..')  # so we can import from the project root

from dotenv import load_dotenv
load_dotenv('../.env')

from deps import get_vectorstore
from langchain_openai import ChatOpenAI
import pandas as pd

## 1. Test Questions
Each question has a reference answer to compare against.

In [ ]:
EVAL_QUESTIONS = [
    {
        "question": "What are the HbA1c targets for type 2 diabetes?",
        "reference": "HbA1c target is generally below 7% for most adults with type 2 diabetes."
    },
    {
        "question": "What is the first-line medication for type 2 diabetes?",
        "reference": "Metformin is the recommended first-line medication for type 2 diabetes."
    },
    {
        "question": "How should hypertension be managed in patients with diabetes?",
        "reference": "Blood pressure targets for diabetic patients are typically below 130/80 mmHg, with ACE inhibitors or ARBs as preferred agents."
    },
    {
        "question": "What are the symptoms of hypoglycemia?",
        "reference": "Symptoms include sweating, trembling, confusion, rapid heartbeat, and in severe cases loss of consciousness."
    },
    {
        "question": "What lifestyle changes help manage type 2 diabetes?",
        "reference": "Diet modification, regular physical activity, weight loss, and smoking cessation are key lifestyle interventions."
    },
]

## 2. Retrieve Chunks from Vectorstore

In [ ]:
vs = get_vectorstore()

def retrieve(question, k=3):
    docs = vs.similarity_search(question, k=k)
    return "\n\n".join(doc.page_content for doc in docs)

print("Vectorstore loaded.")

## 3. Generate Answers

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def generate_answer(question, context):
    prompt = f"""You are a diabetes clinical assistant. Answer the question using only the context below.
If the context does not contain enough information, say so.

Context:
{context}

Question: {question}
Answer:"""
    return llm.invoke(prompt).content

results = []
for item in EVAL_QUESTIONS:
    context = retrieve(item["question"])
    answer  = generate_answer(item["question"], context)
    results.append({
        "question":  item["question"],
        "reference": item["reference"],
        "answer":    answer,
        "context":   context,
    })
    print(f"Done: {item['question'][:60]}...")

print("\nAll answers generated.")

## 4. LLM as Judge
Score each answer 1-5 for faithfulness and relevance.

In [ ]:
judge = ChatOpenAI(model="gpt-4o", temperature=0)

JUDGE_PROMPT = """You are evaluating a RAG system for a diabetes chatbot.

Question: {question}
Reference answer: {reference}
Generated answer: {answer}

Score the generated answer on two dimensions (1-5 each):
- Faithfulness: Is the answer grounded in the reference? No hallucinations?
- Relevance: Does it actually answer the question?

Reply in this exact format:
FAITHFULNESS: <1-5>
RELEVANCE: <1-5>
REASON: <one sentence>"""

def judge_answer(item):
    response = judge.invoke(JUDGE_PROMPT.format(**item)).content
    lines = response.strip().split("\n")
    scores = {}
    for line in lines:
        if line.startswith("FAITHFULNESS:"):
            scores["faithfulness"] = int(line.split(":")[1].strip())
        elif line.startswith("RELEVANCE:"):
            scores["relevance"] = int(line.split(":")[1].strip())
        elif line.startswith("REASON:"):
            scores["reason"] = line.split(":", 1)[1].strip()
    return scores

for item in results:
    scores = judge_answer(item)
    item.update(scores)

print("Judging complete.")

## 5. Results

In [ ]:
df = pd.DataFrame(results)[["question", "faithfulness", "relevance", "reason", "answer"]]
pd.set_option("display.max_colwidth", 80)
display(df)

print(f"\nAvg Faithfulness: {df['faithfulness'].mean():.2f}/5")
print(f"Avg Relevance:    {df['relevance'].mean():.2f}/5")

## 6. Build a Correctness Dataset

A **correctness dataset** pairs each question with a ground-truth reference answer.
This is the foundation for repeatable evals — every future run of the RAG pipeline
is scored against the same gold standard.

Each example has:
- `inputs`: the question sent to the RAG system
- `outputs`: the expected (reference) answer

Once uploaded to LangSmith you can run `evaluate()` against this dataset from any
notebook or CI job without re-defining the questions.

In [ ]:
# Build the correctness dataset from the same questions used above
correctness_dataset = [
    {"inputs": {"question": item["question"]}, "outputs": {"answer": item["reference"]}}
    for item in EVAL_QUESTIONS
]

print(f"{len(correctness_dataset)} examples ready")
for ex in correctness_dataset:
    print(f"  Q: {ex['inputs']['question'][:70]}")
    print(f"  A: {ex['outputs']['answer'][:70]}\n")

## 7. Upload Correctness Dataset to LangSmith

Requires `LANGCHAIN_API_KEY` in your `.env` file (get it from smith.langchain.com → Settings → API Keys).

```
LANGCHAIN_API_KEY=ls__...
```

`create_dataset` is idempotent when you pass `data_type="kv"` — re-running this cell
won't duplicate the dataset, it will raise if the name already exists so the
`try/except` below handles that gracefully.

In [ ]:
import os
from langsmith import Client

ls = Client()  # reads LANGCHAIN_API_KEY from env automatically

DATASET_NAME = "medical-rag-correctness"

# Create dataset (skip if it already exists)
try:
    dataset = ls.create_dataset(
        dataset_name=DATASET_NAME,
        description="Ground-truth Q&A pairs for DiabetesAssist RAG correctness evaluation",
        data_type="kv",
    )
    print(f"Created dataset: {dataset.name} ({dataset.id})")
except Exception as e:
    dataset = ls.read_dataset(dataset_name=DATASET_NAME)
    print(f"Dataset already exists: {dataset.name} ({dataset.id})")

# Upload examples
ls.create_examples(
    inputs=[ex["inputs"] for ex in correctness_dataset],
    outputs=[ex["outputs"] for ex in correctness_dataset],
    dataset_id=dataset.id,
)

print(f"\nUploaded {len(correctness_dataset)} examples to LangSmith.")
print(f"View at: https://smith.langchain.com/datasets/{dataset.id}")

### Verify the upload

List the examples back from LangSmith to confirm everything landed correctly.

In [ ]:
examples = list(ls.list_examples(dataset_id=dataset.id))
print(f"{len(examples)} examples in LangSmith dataset '{DATASET_NAME}':\n")
for ex in examples:
    print(f"  [{ex.id}]")
    print(f"    input : {ex.inputs['question'][:70]}")
    print(f"    output: {ex.outputs['answer'][:70]}\n")